In [1]:
#test1
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# D2Q9 discrete Velocity Vectors: [cx, cy]

c = np.array([
    [0,0], # 0: rest
    [1,0], #1: east
    [0,1], #2: north
    [-1,0], #3: west
    [0,-1], #4: south
    [1,1], #5: northeast
    [-1,1], #6: northwest
    [-1,-1], #7: southwest
    [1,-1]  #8: southeast
])

c

array([[ 0,  0],
       [ 1,  0],
       [ 0,  1],
       [-1,  0],
       [ 0, -1],
       [ 1,  1],
       [-1,  1],
       [-1, -1],
       [ 1, -1]])

In [ ]:
# D2Q9 weights 
# hosen so that the D2Q9 lattice correctly reproduces the isotropic behaviour needed to recover incompressible Navier–Stokes flow at low Mach number

w = np.array([
    4/9, #0: rest
    1/9, #1: east
    1/9, #2: north
    1/9, #3: west
    1/9, #4: south
    1/36, #5: northeast
    1/36, #6: northwest
    1/36, #7: southwest
    1/36  #8: southeast
])
w
print("sum of weights = ", np.sum(w))

sum of weights =  1.0


In [4]:
# define popilations f_i

# One lattice node: nine directional populations
f = np.array([
    4/9,    # f0: rest
    1/9,    # f1: east
    1/9,    # f2: north
    1/9,    # f3: west
    1/9,    # f4: south
    1/36,   # f5: north-east
    1/36,   # f6: north-west
    1/36,   # f7: south-west
    1/36,   # f8: south-east
])

f
print("Original f:")
print(f)
print("Shape:", f.shape)

print("\nf[:, None]:")
print(f[:, None])
print("Shape:", f[:, None].shape)

Original f:
[0.44444444 0.11111111 0.11111111 0.11111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Shape: (9,)

f[:, None]:
[[0.44444444]
 [0.11111111]
 [0.11111111]
 [0.11111111]
 [0.11111111]
 [0.02777778]
 [0.02777778]
 [0.02777778]
 [0.02777778]]
Shape: (9, 1)


In [5]:
# Density = sum of populations at a lattice node
rho = np.sum(f)

# momentum = sum of populations multiplied by their respective velocity vectors
momentum = np.sum(f[:, None] * c, axis=0)
# [:,none] - the none is used to add another dimension - is used to convert the 1D array f into a 2D column vector so that it can be multiplied element-wise with the 2D array c. The result is a 2D array where each row corresponds to a population and its associated velocity vector. The np.sum(..., axis=0) then sums over the rows to give the total momentum in each direction (x and y).

# axis =0 - specifies that the summation should be performed along the first axis (rows) of the resulting 2D array. This means that we are summing the contributions of all populations to get the total momentum in each direction.

# note we are not doin matrix multiplication here, we are doing element-wise multiplication and then summing the results to get the total momentum in each direction.

# average velocity = momentum / density
u = momentum / rho

print("Density:", rho)
print("Momentum:", momentum)
print("Velocity:", u)

Density: 1.0
Momentum: [0. 0.]
Velocity: [0. 0.]


In [7]:
#create a tiny, controlled rightward flow at one lattice node.

#We will increase the east-going population and decrease the west-going population by the same amount. This preserves density, but creates positive x-momentum.

# Start again from the stationary equilibrium state
# Make a new array named f by copying w.
# w contains the equilibrium D2Q9 weights.
# .copy() is important: it gives f its own separate values,
# so changing f later does not alter w.
f = w.copy()


# Store 0.02 in a variable named delta.
# This is the amount of population we will transfer
# from the west-going direction to the east-going direction.
delta = 0.02


# f[1] is the east-going population because c[1] = [1, 0].
# Add delta, so slightly more population travels east.
f[1] = f[1] + delta


# f[3] is the west-going population because c[3] = [-1, 0].
# Subtract the same delta, so slightly less population travels west.
# Adding and subtracting the same amount preserves total density.
f[3] = f[3] - delta


# print() displays information but does not change any values.
# This displays the complete modified array f.
print("Modified f:", f)


# np.sum(f) adds all nine populations f[0] through f[8].
# Store that total in rho, the local density.
rho = np.sum(f)


# Display the density. It should still be 1.0.
print("Density:", rho)

Modified f: [0.44444444 0.13111111 0.11111111 0.09111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Density: 0.9999999999999999


In [8]:
# Create a NumPy array with two zeros:
# first position = total x-momentum
# second position = total y-momentum
momentum = np.array([0.0, 0.0])


# range(9) produces the direction indices:
# 0, 1, 2, 3, 4, 5, 6, 7, 8.
# The loop visits every D2Q9 direction once.
for i in range(9):

    # f[i] is the population travelling in direction i.
    # c[i] is that direction's vector [cx, cy].
    # f[i] * c[i] calculates this population's x- and y-momentum contribution.
    #
    # Example for i = 1:
    # f[1] * c[1] = f[1] * [1, 0]
    #
    # Add that contribution to the running total momentum.
    momentum = momentum + f[i] * c[i]


# Momentum equals rho * velocity.
# Divide both x- and y-momentum by density rho
# to calculate the local velocity vector u = [ux, uy].
u = momentum / rho


# Display the final momentum vector [x-momentum, y-momentum].
print("Momentum:", momentum)


# Display the final velocity vector [ux, uy].
print("Velocity:", u)

Momentum: [0.04 0.  ]
Velocity: [0.04 0.  ]


In [ ]:
# Set the density we want at this lattice node.
# In lattice units, rho = 1.0 is the usual convenient starting value.
rho_target = 1.0

# Create a two-element NumPy array named u_target, initialized to zeros.
# This represents the target velocity vector [ux, uy] at this lattice node. 
# Both components are initially set to zero, indicating no flow in either direction.

u_target = np.array([0.0, 0.0])

# Multiply every D2Q9 weight by the target density to get the equilibrium populations.
# This is done using element-wise multiplication of the weight array w with the scalar rho_target.
#  f_eq[0], f_eq[1], ..., f_eq[8] are the equilibrium populations corresponding to each of the nine discrete velocity directions in the D2Q9 model.
f_eq = rho_target * w

# Display the nine equilibrium populations.
print("Equilibrium populations:", f_eq)
#note that the original equation of F_eq = rho * w * (1 + 3 * (c @ u) + 9/2 * (c @ u)**2 - 3/2 * (u @ u)) is a more general form that accounts for non-zero velocities. In this case, since we are setting the target velocity to zero, the simplified version f_eq = rho_target * w is sufficient to represent the equilibrium state at rest.
# in current case we have chosen ideal values u = [0,0] where everything els becomes 1



# Add all nine equilibrium populations.
# This checks whether they recover the density we prescribed.
rho_check = np.sum(f_eq)


# Display the recovered density.
print("Recovered density:", rho_check)



Equilibrium populations: [0.44444444 0.11111111 0.11111111 0.11111111 0.11111111 0.02777778
 0.02777778 0.02777778 0.02777778]
Recovered density: 1.0
